# Auto-graded Task 2: Movie Recommendation System

This notebook completes Auto-graded Task 2 of *NLP – Semantic Similarity*. It reads the supplied `movies.txt` file, compares the movie descriptions with the plot of *Planet Hulk*, and returns the title of the most similar movie.

## 1. Environment setup (run these cells first)

The notebook needs **spaCy** and its **`en_core_web_md`** language model, which includes pretrained word vectors. Run the following setup cells in order before the rest of the notebook. An internet connection is needed for the initial downloads. The cells skip installation if the requirements are already available.

**After installing packages or a language model**, restart the notebook kernel if an import fails or a restart is requested, then run the cells again from the top. Installation must be repeated in a new temporary hosted environment (for example, a new Colab runtime).

In [ ]:
# Install spaCy into the Python environment used by this notebook, if needed.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "spacy"
    ])
    print("spaCy installed. Restart the kernel if prompted.")
else:
    print("spaCy is already installed.")

spaCy is already installed.


In [ ]:
# Install the medium-sized English model in this notebook's environment.
import spacy

try:
    spacy.load("en_core_web_md")
    print("en_core_web_md is already available.")
except OSError:
    subprocess.check_call([
        sys.executable, "-m", "spacy", "download", "en_core_web_md"
    ])
    print("Model downloaded. Restart the kernel if loading still fails.")

Model downloaded. Restart the kernel if loading still fails.


In [ ]:
# Verify the installed library and model before proceeding.
import spacy

nlp = spacy.load("en_core_web_md")
print("spaCy version:", spacy.__version__)
print("Language model:", nlp.meta["name"])
print("Word vector dimensions:", nlp.vocab.vectors_length)
assert nlp.vocab.vectors_length > 0, (
    "The loaded language model does not contain word vectors."
)
print("Environment setup completed successfully.")

spaCy version: 3.8.18
Language model: core_web_md
Word vector dimensions: 300
Environment setup completed successfully.


## 2. Import spaCy and load the language model

The model was loaded and verified in the setup section above. The cell below makes the model choice explicit for the main solution.

In [ ]:
import spacy

# Load the medium-sized English model for semantic comparisons.
nlp = spacy.load("en_core_web_md")
print("Language model loaded:", nlp.meta["name"])

Language model loaded: core_web_md


## 3. Define the Planet Hulk description

The reference description is taken directly from the practical task.

In [ ]:
PLANET_HULK_DESCRIPTION = (
    "Will he save their world or destroy it? When the Hulk becomes too "
    "dangerous for the Earth, the Illuminati trick Hulk into a shuttle "
    "and launch him into space to a planet where the Hulk can live in "
    "peace. Unfortunately, Hulk lands on the planet Sakaar where he is "
    "sold into slavery and trained as a gladiator."
)

## 4. Read the supplied movie descriptions

Each line of `movies.txt` contains a movie label, a colon, and a description. The code separates these into a dictionary so that the descriptions can be compared and the corresponding labels can be returned.

###Accessing movies.txt in Your Notebook Environment

Before running the cell that loads the movie descriptions, ensure that the movies.txt file provided with the task is accessible in your notebook environment. Follow the instructions for your chosen platform.

Google Colab
1. Click the Files icon (folder symbol) in the left sidebar.
2. Select Upload to session storage.
3. Upload the movies.txt file provided with the task.

Note: Files uploaded to Google Colab's session storage are temporary. If the runtime restarts or disconnects, the file may need to be uploaded again.

Jupyter Notebook
1. Save movies_task.ipynb and movies.txt in the same folder on your computer.
2. Open Jupyter Notebook and navigate to the folder containing both files.
3. Open movies_task.ipynb.

Visual Studio Code (VS Code)
1. Create a folder on your computer for the task.
2. Save movies_task.ipynb and movies.txt inside this folder.
3. Open VS Code and select File → Open Folder.
4. Select the folder containing both files.
5. Open movies_task.ipynb and select a Python kernel when prompted.

In [ ]:
from pathlib import Path


def load_movies(file_path="movies.txt"):
    """Read movie labels and descriptions from the supplied text file.

    Args:
        file_path (str): Path to the file containing movie descriptions.

    Returns:
        dict: Movie labels mapped to their plot descriptions.
    """
    movies = {}

    # Open the supplied file using UTF-8 text encoding.
    with Path(file_path).open("r", encoding="utf-8") as movie_file:
        for line in movie_file:
            if not line.strip():
                continue

            # Split only on the first colon, preserving any in the plot.
            title, separator, description = line.partition(":")
            if separator and title.strip() and description.strip():
                movies[title.strip()] = description.strip()

    if not movies:
        raise ValueError("No valid movie descriptions were found.")

    return movies


movies = load_movies()
print(f"Loaded {len(movies)} movies.")
print("Movie labels:", ", ".join(movies))

Loaded 10 movies.
Movie labels: Movie A, Movie B, Movie C, Movie D, Movie E, Movie F, Movie G, Movie H, Movie I, Movie J


## 5. Create the recommendation function

The function accepts a **movie description as its parameter**, as required by the task. It compares that description with every movie in the supplied file and returns the label of the movie with the highest similarity score.

In [ ]:
def recommend_movie(description):
    """Return the label of the movie most similar to a description.

    Args:
        description (str): Plot description of the watched movie.

    Returns:
        str: Label of the movie with the highest similarity score.
    """
    # Process the watched movie description using spaCy.
    reference = nlp(description)
    best_title = None
    highest_similarity = float("-inf")

    # Compare the watched movie with every available movie.
    for title, plot in movies.items():
        similarity = reference.similarity(nlp(plot))

        # Retain the title with the strongest similarity score.
        if similarity > highest_similarity:
            highest_similarity = similarity
            best_title = title

    return best_title

## 6. Run the recommendation system

The returned label identifies the movie whose description is most similar to the *Planet Hulk* description according to the loaded spaCy model.

In [ ]:
recommended_title = recommend_movie(PLANET_HULK_DESCRIPTION)
print(f"Recommended movie: {recommended_title}")

Recommended movie: Movie F


## 7. Optional: Inspect the similarity scores

This additional cell makes the recommendation easier to verify. It is not necessary for the function to work, but it shows how the most similar movie was selected.

In [ ]:
# Display the similarity scores in descending order.
reference = nlp(PLANET_HULK_DESCRIPTION)
scores = []

for title, plot in movies.items():
    score = reference.similarity(nlp(plot))
    scores.append((title, score))

for title, score in sorted(scores, key=lambda item: item[1], reverse=True):
    print(f"{title}: {score:.3f}")

Movie F: 0.952
Movie C: 0.949
Movie A: 0.936
Movie H: 0.927
Movie B: 0.920
Movie G: 0.918
Movie J: 0.899
Movie I: 0.875
Movie E: 0.850
Movie D: 0.723
